<a href="https://colab.research.google.com/github/adhikaryramen87/MachineLearning_Works/blob/main/GenAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install git+https://github.com/openai/CLIP.git

import torch
import clip
import numpy as np
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------
# 1. Load CLIP
# ---------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

# ---------------------------------------------------
# 2. Image Embedding (normalized)
# ---------------------------------------------------
def get_image_embedding(img_path):
    image = Image.open(img_path).convert("RGB")
    image_tensor = preprocess(image).unsqueeze(0).to(device)

    with torch.no_grad():
        emb = model.encode_image(image_tensor)

    emb = emb / emb.norm(dim=-1, keepdim=True)  # CRITICAL
    return emb.cpu().numpy()

# ---------------------------------------------------
# 3. CLIP-style Softmax Similarity
# ---------------------------------------------------
def clip_similarity(query_emb, candidate_embs, temperature=0.07):
    """
    Returns probabilities (relative similarity)
    """
    sims = cosine_similarity(query_emb, candidate_embs)[0]
    sims = sims / temperature
    exp_sims = np.exp(sims)
    probs = exp_sims / np.sum(exp_sims)
    return probs, sims

# ---------------------------------------------------
# 4. Load images
# ---------------------------------------------------
img_query = "dog_1.jpg"
img_pos = "dog_2.jpg"
img_neg = "airplane.jpg"

emb_query = get_image_embedding(img_query)
emb_pos = get_image_embedding(img_pos)
emb_neg = get_image_embedding(img_neg)

# ---------------------------------------------------
# 5. Compute similarity
# ---------------------------------------------------
candidate_embeddings = np.vstack([emb_pos, emb_neg])
probs, raw_sims = clip_similarity(emb_query, candidate_embeddings)

print("Raw cosine similarities:")
print("Dog vs Dog    :", raw_sims[0])
print("Dog vs Plane  :", raw_sims[1])

print("\nCLIP-style probabilities:")
print("Dog image probability   :", probs[0])
print("Plane image probability :", probs[1])

Raw cosine similarities:

Dog vs Dog    : 12.533122

Dog vs Plane  : 7.2286034


Zero Shot Learning

1. Role: "You are a Safety Auditor".

2. Tasks: Check for blocked exits and missing helmets

3. Format: Return the results in the JSON format.

In [ ]:
# --- THE CONCEPTUAL SAFETY PIPELINE ---

def simulate_safety_brain(image_input, safety_rules):
    """
    A conceptual look at how a Vision-Language Model (VLM)
    processes an image into a structured safety decision.
    """

    # 1. THE INPUT: The model receives raw pixels + your instructions.
    print(f"DEBUG: Processing image {image_input}...")

    # 2. THE FEATURE EXTRACTION:
    # The model turns pixels into 'Meaning Vectors' (what we learned in Sec 1).
    # It identifies 'Rectangular Obstacle' near 'Red Exit Sign'.

    # 3. THE REASONING (The 'Zero-Shot' Magic):
    # The model compares the 'Meaning Vectors' against the 'Safety Rules'.
    # 'If Obstacle is in front of Exit -> Hazard = True'

    # 4. THE OUTPUT: A structured response for the manager.
    demo_report = {
        "timestamp": "2026-03-10 14:00",
        "location": "Aisle 4, Zone B",
        "risk_found": True,
        "details": "Two wooden pallets are obstructing the emergency fire exit.",
        "risk_level": "CRITICAL",
        "action_item": "Clear the exit path immediately."
    }

    return demo_report

# --- EXECUTION ---
# report = simulate_safety_brain("warehouse_cam_01.jpg", "Find blocked exits")
# print(report["details"])